# jax.lax — Control Flow and Low-Level Primitives

This notebook explores `jax.lax`, JAX's library of low-level operations that work correctly inside `jit`.

### Why can't we just use normal Python?

When JAX traces your function for `jit`, it doesn't **run** your Python code — it **records** what operations happen. This means:

```python
# ❌ This breaks inside jit:
if x > 0:       # Python needs a concrete True/False at trace time
    return x
else:
    return -x

# ✅ This works inside jit:
jax.lax.cond(x > 0, lambda x: x, lambda x: -x, x)
```

**Rule of thumb:** If your code has `if`, `for`, or `while` that depends on array **values**, use `jax.lax` equivalents.

In [ ]:
# Import JAX libraries
import jax
import jax.numpy as jnp
import jax.lax as lax

### ✅ Exercise 71 — lax.cond (If-Else for JIT)

Python's `if` checks a condition **once** at trace time. `lax.cond` checks it **every time** the function runs.

```python
lax.cond(condition, true_fn, false_fn, operand)
```
- If `condition` is True → calls `true_fn(operand)`
- If `condition` is False → calls `false_fn(operand)`

Both branches must return the **same shape and dtype**.

In [ ]:
# Absolute value using lax.cond — works inside jit
@jax.jit
def safe_abs(x):
    return lax.cond(
        x >= 0,
        lambda x: x,      # true branch
        lambda x: -x,     # false branch
        x
    )

print(safe_abs(jnp.float32(5.0)))   # → 5.0
print(safe_abs(jnp.float32(-3.0)))  # → 3.0

### ✅ Exercise 72 — lax.switch (Multi-Branch Selection)

`lax.switch` is like a `match`/`switch` statement — pick one of N branches based on an integer index.

```python
lax.switch(index, [fn_0, fn_1, fn_2, ...], operand)
```

If `index=1`, it calls `fn_1(operand)`. All branches must return the same shape.

In [ ]:
# Choose an activation function at runtime
@jax.jit
def apply_activation(x, choice):
    return lax.switch(
        choice,
        [
            lambda x: jnp.tanh(x),         # 0: tanh
            lambda x: jax.nn.relu(x),       # 1: relu
            lambda x: jax.nn.sigmoid(x),    # 2: sigmoid
        ],
        x
    )

x = jnp.float32(2.0)
print(f"tanh:    {apply_activation(x, 0):.4f}")
print(f"relu:    {apply_activation(x, 1):.4f}")
print(f"sigmoid: {apply_activation(x, 2):.4f}")

### ✅ Exercise 73 — lax.fori_loop (Fixed-Count Loop)

Python `for i in range(n)` works in JIT only if `n` is a **constant**. `lax.fori_loop` works even when the loop count depends on data.

```python
lax.fori_loop(lower, upper, body_fn, init_val)
```
- Loops from `lower` to `upper - 1`
- `body_fn(i, val)` takes the loop index and current value, returns new value
- Think of it as: `val = body_fn(0, val); val = body_fn(1, val); ...`

In [ ]:
# Sum numbers 0 to 9 using fori_loop
@jax.jit
def sum_range(n):
    return lax.fori_loop(
        0, n,
        lambda i, acc: acc + i,  # body: add current index to accumulator
        0                        # initial value
    )

print(sum_range(10))   # → 45  (0+1+2+...+9)
print(sum_range(100))  # → 4950

### ✅ Exercise 74 — lax.while_loop (Conditional Loop)

`while_loop` keeps going until a condition becomes False. Unlike `fori_loop`, the number of iterations isn't known ahead of time.

```python
lax.while_loop(cond_fn, body_fn, init_val)
```
- `cond_fn(val)` → True to keep looping, False to stop
- `body_fn(val)` → returns the next value
- `init_val` → starting state

In [ ]:
# Find smallest power of 2 >= target
@jax.jit
def next_power_of_2(target):
    # State is (current_value,)
    init = jnp.array(1, dtype=jnp.int32)
    result = lax.while_loop(
        lambda val: val < target,   # keep going while val < target
        lambda val: val * 2,        # double each iteration
        init
    )
    return result

print(next_power_of_2(jnp.int32(5)))    # → 8
print(next_power_of_2(jnp.int32(100)))  # → 128
print(next_power_of_2(jnp.int32(1024))) # → 1024

### ✅ Exercise 75 — lax.scan (The Most Powerful Loop)

`scan` is like `fori_loop` but it **collects outputs** at each step. This is essential for:
- RNNs (carry hidden state, collect outputs)
- Time series (process each timestep)
- Any sequential computation

```python
final_carry, stacked_outputs = lax.scan(fn, init_carry, xs)
```
- `fn(carry, x)` → `(new_carry, output)` — processes one element
- `xs` — array to scan over (like a batch of timesteps)
- Returns: final carry + all outputs stacked

In [ ]:
# Cumulative sum using scan
# At each step: carry = running total, output = running total so far
def cumsum_step(carry, x):
    new_carry = carry + x
    return new_carry, new_carry  # (next carry, output for this step)

xs = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
final_total, running_totals = lax.scan(cumsum_step, 0.0, xs)

print(f"Input:          {xs}")
print(f"Running totals: {running_totals}")  # → [1, 3, 6, 10, 15]
print(f"Final total:    {final_total}")      # → 15

### ✅ Exercise 76 — scan for an RNN-Style Computation

This is how sequence models work in JAX. The hidden state is the **carry**, and at each timestep we produce an output.

```
h₀ → [step] → h₁ → [step] → h₂ → [step] → h₃
       ↓              ↓              ↓
      y₁             y₂             y₃
```

In [ ]:
# Simple RNN cell: h_new = tanh(W_h @ h + W_x @ x)
def rnn_step(carry, x):
    h, params = carry
    W_h, W_x = params
    h_new = jnp.tanh(W_h @ h + W_x @ x)
    return (h_new, params), h_new  # carry params through, output hidden state

# Initialize
key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key)
hidden_size, input_size = 4, 3
W_h = jax.random.normal(k1, (hidden_size, hidden_size)) * 0.1
W_x = jax.random.normal(k2, (hidden_size, input_size)) * 0.1

h0 = jnp.zeros(hidden_size)          # initial hidden state
seq = jax.random.normal(key, (5, 3))  # 5 timesteps, 3 features each

# Run the RNN over the sequence
(h_final, _), all_hiddens = lax.scan(rnn_step, (h0, (W_h, W_x)), seq)

print(f"Sequence length: {seq.shape[0]}")
print(f"Hidden states:   {all_hiddens.shape}")  # (5, 4)
print(f"Final hidden:    {h_final}")

### ✅ Exercise 77 — lax.select (Element-Wise If-Else)

`lax.select` is like `jnp.where` — it picks elements from two arrays based on a boolean mask. Unlike `lax.cond` which picks one **branch**, `select` picks individual **elements**.

```python
lax.select(condition_array, on_true_array, on_false_array)
```

In [ ]:
# Clamp values: replace negatives with 0 (like ReLU)
@jax.jit
def manual_relu(x):
    return lax.select(
        x > 0,             # condition (per-element)
        x,                 # where True: keep x
        jnp.zeros_like(x)  # where False: use 0
    )

x = jnp.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print(f"Input:  {x}")
print(f"Output: {manual_relu(x)}")

### ✅ Exercise 78 — lax.clamp

Clips values to a range `[min, max]`. Simple but very common in practice — gradient clipping, value normalization, etc.

```python
lax.clamp(min_val, x, max_val)  # min ≤ result ≤ max
```

In [ ]:
@jax.jit
def clip_gradients(grads, max_norm):
    return lax.clamp(-max_norm, grads, max_norm)

grads = jnp.array([-5.0, -0.5, 0.3, 2.0, 10.0])
clipped = clip_gradients(grads, jnp.float32(1.0))

print(f"Original: {grads}")
print(f"Clipped:  {clipped}")  # values clamped to [-1, 1]

### ✅ Exercise 79 — scan for Training (Multiple Steps)

Instead of a Python `for` loop for training steps, use `scan`. This is:
- **JIT-compilable** as a single fused operation
- **Memory efficient** — XLA can optimize across steps
- The idiomatic JAX way to run training loops

In [ ]:
# Simple linear regression: y = w*x + b
def loss_fn(params, x, y):
    w, b = params
    pred = w * x + b
    return jnp.mean((pred - y) ** 2)

def train_step(params, _):
    grads = jax.grad(loss_fn)(params, xs, ys)
    params = jax.tree.map(lambda p, g: p - 0.01 * g, params, grads)
    loss = loss_fn(params, xs, ys)
    return params, loss  # carry=params, output=loss

# Data: y = 3x + 1
key = jax.random.PRNGKey(0)
xs = jax.random.normal(key, (100,))
ys = 3.0 * xs + 1.0

# Initial params
params = (jnp.float32(0.0), jnp.float32(0.0))  # (w, b)

# Run 200 training steps with scan
final_params, losses = jax.jit(lambda p: lax.scan(train_step, p, None, length=200))(params)

w, b = final_params
print(f"Learned: y = {w:.4f}x + {b:.4f}")
print(f"Target:  y = 3.0000x + 1.0000")
print(f"Final loss: {losses[-1]:.6f}")

### ✅ Exercise 80 — Combining lax Primitives

Let's put it all together: use `scan` with `cond` inside to build a sequence processor that behaves differently based on input values.

This pattern appears everywhere — masked attention, gated RNNs, conditional computation.

In [ ]:
# Gated accumulator: if x > 0, add x to running sum; otherwise, halve the sum
def gated_step(carry, x):
    new_carry = lax.cond(
        x > 0,
        lambda c: c + x,     # gate open: accumulate
        lambda c: c * 0.5,   # gate closed: decay
        carry
    )
    return new_carry, new_carry

xs = jnp.array([1.0, 2.0, -1.0, 3.0, -1.0, 4.0])
final, history = jax.jit(lambda xs: lax.scan(gated_step, 0.0, xs))(xs)

print(f"Input:   {xs}")
print(f"History: {history}")
print(f"Final:   {final}")
# Step by step:
# x=1  → 0 + 1 = 1.0
# x=2  → 1 + 2 = 3.0
# x=-1 → 3 * 0.5 = 1.5
# x=3  → 1.5 + 3 = 4.5
# x=-1 → 4.5 * 0.5 = 2.25
# x=4  → 2.25 + 4 = 6.25

## Conclusion

### When to Use What

| Python | jax.lax equivalent | Use when... |
|---|---|---|
| `if/else` | `lax.cond` | Condition depends on array values |
| `match/switch` | `lax.switch` | Multi-way branch on an index |
| `for i in range(n)` | `lax.fori_loop` | Fixed loop, don't need outputs |
| `for x in xs` | `lax.scan` | Sequential, need carry + outputs |
| `while` | `lax.while_loop` | Unknown number of iterations |
| `np.where` | `lax.select` | Element-wise conditional |
| `np.clip` | `lax.clamp` | Clamp to range |

### The key insight

Python control flow runs at **trace time** (once). `jax.lax` control flow runs at **execution time** (every call). If your logic depends on runtime array values and needs to work inside `jit`, use `jax.lax`.